# Session 5 — GPU: Precision / Dtype Sweep (FP32 / FP16 / BF16 / INT8)

## Background

Sessions 1–4 ran exclusively in FP32. Modern accelerators expose lower-precision math units that deliver higher FLOP/s at reduced memory footprint: NVIDIA Tensor Cores accelerate FP16 and BF16 matrix multiplications; TPU MXUs are native BF16. The tradeoff is numerical range and precision. FP16 has a narrow dynamic range and can underflow during training (addressed by loss scaling). BF16 shares FP32's exponent range, making it drop-in safe for most training workloads without a scaler.

INT8 quantization goes further: 8-bit integer arithmetic at ~2× the throughput of BF16 and ~50% VRAM reduction. On the NVIDIA L4, `torch.ao.quantization` provides dynamic INT8 quantization for inference. The v5e TPU supports native INT8 MXU operations (786 TOPS = 2× BF16 TFLOPS).

## Goal

Measure how much throughput FP16, BF16, and INT8 gain over FP32 on the NVIDIA L4 across six batch sizes. Track peak VRAM to confirm the expected memory reduction.

## Question

How much throughput does each precision level add on the GPU relative to FP32, and how does the speedup depend on batch size?

---

**Runtime:** GPU (NVIDIA L4 or equivalent)

After running, results are saved to `results/gpu_dtype_sweep.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
import time, torch, torch.nn as nn

assert torch.cuda.is_available(), "No GPU found. Runtime → Change runtime type → GPU."

device = torch.device("cuda")
print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print(f"BF16 supported : {torch.cuda.is_bf16_supported()}")

Device  : NVIDIA L4
VRAM    : 23.7 GB
PyTorch : 2.10.0+cu128
BF16 supported : True


In [ ]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 5 config ──────────────────────────────────────────────────────────
SEQ_LEN     = 128
BATCH_SIZES = [32, 64, 128, 256, 512, 1024]
DTYPES_GPU  = ["fp32", "fp16", "bf16"]

# Large batches have long backward passes — use fewer steps to avoid hangs.
STEPS_FOR_BATCH = {32: 50, 64: 50, 128: 50, 256: 50, 512: 20, 1024: 20}
WARMUP = 5

DTYPE_MAP = {
    "fp32": torch.float32,
    "fp16": torch.float16,
    "bf16": torch.bfloat16,
}

def benchmark(batch_size, dtype_str, device):
    dtype   = DTYPE_MAP[dtype_str]
    use_amp = dtype_str != "fp32"
    n_steps = STEPS_FOR_BATCH[batch_size]

    model     = BenchmarkTransformerBlock(
                    d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD
                ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()

    scaler = torch.amp.GradScaler("cuda") if dtype_str == "fp16" else None

    torch.cuda.reset_peak_memory_stats(device)

    elapsed = 0.0
    for step in range(n_steps + WARMUP):
        x = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)

        torch.cuda.synchronize()
        t0 = time.perf_counter()

        optimizer.zero_grad()
        if use_amp:
            with torch.autocast("cuda", dtype=dtype):
                loss = criterion(model(x), y)
        else:
            loss = criterion(model(x), y)

        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        torch.cuda.synchronize()
        t1 = time.perf_counter()

        if step >= WARMUP:
            elapsed += t1 - t0

    throughput   = (n_steps * batch_size) / elapsed
    latency_ms   = (elapsed / n_steps) * 1000
    peak_vram_mb = torch.cuda.max_memory_allocated(device) / 1e6
    torch.cuda.empty_cache()
    return round(throughput, 1), round(latency_ms, 2), round(peak_vram_mb, 1)

print(f"Config  : seq_len={SEQ_LEN}, d_model={D_MODEL}")
print(f"Dtypes  : {DTYPES_GPU}")
print(f"Batches : {BATCH_SIZES}  (steps: {STEPS_FOR_BATCH})")
print("Benchmark function defined.")

Config  : seq_len=128, d_model=512
Dtypes  : ['fp32', 'fp16', 'bf16']
Batches : [32, 64, 128, 256, 512, 1024]  (steps: {32: 50, 64: 50, 128: 50, 256: 50, 512: 20, 1024: 20})
Benchmark function defined.


In [ ]:
results = {d: {} for d in DTYPES_GPU}

for dtype_str in DTYPES_GPU:
    print(f"\n── {dtype_str.upper()} ──────────────────────────────────────")
    for bs in BATCH_SIZES:
        n = STEPS_FOR_BATCH[bs]
        print(f"  batch={bs:<6} ({n} steps) ... ", end="", flush=True)
        try:
            tput, lat, vram = benchmark(bs, dtype_str, device)
            results[dtype_str][str(bs)] = {
                "throughput":   tput,
                "latency_ms":   lat,
                "peak_vram_mb": vram,
            }
            print(f"{tput:>10,.0f} samples/sec  {lat:6.1f} ms  VRAM={vram:,.0f} MB")
        except torch.cuda.OutOfMemoryError:
            print("OOM")
            results[dtype_str][str(bs)] = None
            torch.cuda.empty_cache()

print("\n── Speedup vs FP32 ─────────────────────────────────")
header = f"{'batch':<8}" + "".join(f" {d:>10}" for d in DTYPES_GPU)
print(header)
for bs in BATCH_SIZES:
    fp32_val = results["fp32"].get(str(bs))
    if fp32_val is None:
        continue
    row = f"{bs:<8}"
    for d in DTYPES_GPU:
        r = results[d].get(str(bs))
        if r:
            row += f" {r['throughput'] / fp32_val['throughput']:>10.3f}×"
        else:
            row += f" {'OOM':>10}"
    print(row)


── FP32 ──────────────────────────────────────
  batch=32     (50 steps) ...      2,903 samples/sec    11.0 ms  VRAM=296 MB
  batch=64     (50 steps) ...      2,705 samples/sec    23.7 ms  VRAM=531 MB
  batch=128    (50 steps) ...      2,637 samples/sec    48.5 ms  VRAM=980 MB
  batch=256    (50 steps) ...      2,583 samples/sec    99.1 ms  VRAM=1,852 MB
  batch=512    (20 steps) ...      2,579 samples/sec   198.5 ms  VRAM=3,597 MB
  batch=1024   (20 steps) ...      2,524 samples/sec   405.7 ms  VRAM=7,087 MB

── FP16 ──────────────────────────────────────
  batch=32     (50 steps) ...      7,206 samples/sec     4.4 ms  VRAM=233 MB
  batch=64     (50 steps) ...      6,909 samples/sec     9.3 ms  VRAM=400 MB
  batch=128    (50 steps) ...      5,916 samples/sec    21.6 ms  VRAM=715 MB
  batch=256    (50 steps) ...      5,880 samples/sec    43.5 ms  VRAM=1,319 MB
  batch=512    (20 steps) ...      5,822 samples/sec    88.0 ms  VRAM=2,549 MB
  batch=1024   (20 steps) ...      5,497 sample

---

## INT8 Quantization  `# [PENDING RUN]`

INT8 quantization uses 8-bit integers for weights and activations in linear layers, reducing VRAM by ~50% vs FP32 and potentially doubling throughput where memory bandwidth is the bottleneck. On the L4, `torch.ao.quantization` provides dynamic INT8 quantization (weights quantized at load time, activations quantized per-call). For training, INT8 is typically applied to the forward pass only; backward remains in FP32.

**Expected results (L4):**
- Inference throughput: ~1.5–2× vs FP32
- VRAM: ~50% reduction vs FP32
- Training throughput: smaller gain than inference (backward in FP32)

Run the cells below to benchmark. Results will be added to the saved JSON.

In [ ]:
# [PENDING RUN] INT8 dynamic quantization — inference throughput
# torch.ao.quantization.quantize_dynamic converts Linear layers to INT8 at load time.
# Activations are quantized per-call. No calibration dataset needed.

import torch.ao.quantization as quant

def benchmark_int8_inference(batch_size, device, n_steps=50, warmup=5):
    """Benchmark INT8 dynamic quantization — inference only (no backward)."""
    model_fp32 = BenchmarkTransformerBlock(
        d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD
    )
    # Dynamic quantization: Linear layers -> qint8
    # Note: must be on CPU for torch.ao.quantization.quantize_dynamic
    model_int8 = quant.quantize_dynamic(
        model_fp32,
        {nn.Linear},
        dtype=torch.qint8
    )
    model_int8.eval()

    torch.cuda.reset_peak_memory_stats(device)

    elapsed = 0.0
    for step in range(n_steps + warmup):
        # INT8 dynamic quant runs on CPU; move data accordingly
        x = torch.randn(batch_size, SEQ_LEN, D_MODEL)
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model_int8(x)
        if step >= warmup:
            elapsed += time.perf_counter() - t0

    throughput = (n_steps * batch_size) / elapsed
    latency_ms = (elapsed / n_steps) * 1000
    return round(throughput, 1), round(latency_ms, 2)


print("── INT8 (Dynamic Quantization) — Inference throughput ──")
results["int8"] = {}
for bs in BATCH_SIZES:
    print(f"  batch={bs:<6} ... ", end="", flush=True)
    try:
        tput, lat = benchmark_int8_inference(bs, device)
        results["int8"][str(bs)] = {"throughput": tput, "latency_ms": lat, "mode": "inference"}
        print(f"{tput:>10,.0f} samples/sec  {lat:6.1f} ms")
    except Exception as e:
        results["int8"][str(bs)] = None
        print(f"ERROR: {e}")

# Compare INT8 vs FP32 inference
print("\n── INT8 speedup vs FP32 (inference) ─────────────────")
for bs in BATCH_SIZES:
    fp32_r = results["fp32"].get(str(bs))
    int8_r = results["int8"].get(str(bs))
    if fp32_r and int8_r:
        ratio = int8_r["throughput"] / fp32_r["throughput"]
        print(f"  batch={bs:<6}: {ratio:.3f}×")

In [ ]:
import json, os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
payload = {
    "hardware":      "GPU",
    "device_name":   torch.cuda.get_device_name(0),
    "session":       "5",
    "dtypes_tested": list(results.keys()),
    "batch_sizes":   BATCH_SIZES,
    "seq_len":       SEQ_LEN,
    "timestamp":     datetime.now(timezone.utc).isoformat(),
    "results":       results,
}
path = "results/gpu_dtype_sweep.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved → {path}")

Saved → results/gpu_dtype_sweep.json
